In [1]:
# %pip install pandas-3.1.1

In [2]:
# Import libraries

import json
import pandas as pd
import ollama
import autogen

from autogen import ConversableAgent, initiate_chats

In [3]:
# Configure Ollama (the model returns its response all at once or gradually in chunks (stream).)

llm_config = {"config_list": [{"model": "llama3.2", "api_type": "ollama", 
                               "client_host": "http://localhost:11434", "stream": False}] , "temperature": 0.03}

In [4]:
# Create Personal Information Agent

onboarding_personal_information_agent = ConversableAgent(name = "onboarding_personal_information_agent",
                                                         system_message = """
                                                         You are a customer onboarding agent.
                                                         Your responsibility is to collect:
                                                         1. Customer name
                                                         2. Customer location
                                                         Ask the customer for these details.
                                                         Once the customer provides both details,
                                                         respond in this exact format:
                                                         CUSTOMER_DATA:{"name": "customer name", "location": "customer location"}
                                                         Do not invent customer information.
                                                         End your response with TERMINATE.
                                                         """, llm_config = llm_config, code_execution_config = False, 
                                                         human_input_mode = "NEVER")

In [5]:
# Create Topic Preference Agent

onboarding_topic_preference_agent = ConversableAgent(name = "onboarding_topic_preference_agent", 
                                                     system_message = """
                                                     You are a customer topic preference agent.
                                                     Your responsibility is to collect the customer's
                                                     preferred reading topics
                                                     Examples:
                                                     Technology, Sports, Thriller, Business, AI.
                                                     Once the customer provides their preference,
                                                     respond in this format:
                                                     TOPIC_PREFERENCE: customer preference
                                                     Preserve the customer's actual preference.
                                                     Do not invent a different preference.
                                                     End your response with TERMINATE.
                                                     """, llm_config = llm_config, code_execution_config = False, 
                                                     human_input_mode = "NEVER")

In [6]:
# Create Customer Engagement Agent

customer_engagement_agent = ConversableAgent(name="customer_engagement_agent",
                                             system_message="""
                                             You are a customer engagement agent.
                                             Your responsibility is to provide personalized
                                             content based on customer information and preferences.
                                             You will receive:
                                             1. Customer name
                                             2. Customer location
                                             3. Customer topic preference
                                             Follow these instructions:
                                             - Address the customer by their name.
                                             - Understand their actual topic preference.
                                             - Generate relevant and interesting content.
                                             - If the preference is Thriller, provide a short thriller story.
                                             - If the preference is Technology, provide interesting technology information.
                                             - Do not generate unrelated content.
                                             - Never invent customer information.
                                             End your response with TERMINATE.
                                             """, llm_config = llm_config, code_execution_config = False,
                                             human_input_mode = "NEVER",
                                             is_termination_msg = lambda msg: ("terminate" in (msg.get("content") or "").lower()))

In [7]:
# Create Customer Proxy Agent

customer_proxy_agent = ConversableAgent(name="customer_proxy_agent", llm_config = False, 
                                        code_execution_config = False, human_input_mode = "ALWAYS",
                                        is_termination_msg = lambda msg: ("terminate" in (msg.get("content") or "").lower()))

In [8]:
# Create sequential chats

chats = [{"sender": onboarding_personal_information_agent, "recipient": customer_proxy_agent,
          "message": """Hello! Welcome to our platform. Could you tell me your name and location?""",
          "summary_method": "last_msg", "summary_args":{"summary_prompt": """
          Return the customer information as a JSON object only. Example: {"name": "Raji", "location": "Chennai"}"""},
          "max_turns": 2, "clear_history": True},
         
         {"sender": onboarding_topic_preference_agent, "recipient": customer_proxy_agent,
          "message": """Great! Could you tell me what topics, you are interested in reading about?""",
          "summary_method": "last_msg", "max_turns": 2, "clear_history": False},
         
          {"sender": customer_proxy_agent, "recipient": customer_engagement_agent, 
           "message": """Use the customer information and topic preference
           collected in the previous conversations.
           Generate personalized content based on those details.
           Do not use default or invented customer information.""",
           "max_turns": 2, "summary_method": "last_msg"}]

In [9]:
chat_results = initiate_chats(chats)


********************************************************************************
Starting a new chat....

********************************************************************************
onboarding_personal_information_agent (to customer_proxy_agent):

Hello! Welcome to our platform. Could you tell me your name and location?

--------------------------------------------------------------------------------


C:\Users\Raji\anaconda3\envs\ai_agents\Lib\site-packages\autogen\agentchat\chat.py:69: UserWarning: Repetitive recipients detected: The chat history will be cleared by default if a recipient appears more than once. To retain the chat history, please set 'clear_history=False' in the configuration of the repeating agent.
  warnings.warn(


Replying as customer_proxy_agent. Provide feedback to onboarding_personal_information_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  Raji from chennai


customer_proxy_agent (to onboarding_personal_information_agent):

Raji from chennai

--------------------------------------------------------------------------------
onboarding_personal_information_agent (to customer_proxy_agent):

CUSTOMER_DATA:{"name": "Raji", "location": "Chennai"}
TERMINATE

--------------------------------------------------------------------------------


Replying as customer_proxy_agent. Provide feedback to onboarding_personal_information_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  



>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> TERMINATING RUN (04c01dd8-15dc-4db4-bc42-3dc4f1f635b0): Termination message condition on agent 'customer_proxy_agent' met

>>>>>>>> TERMINATING RUN (4bb3faf4-e262-4fcb-9f3c-cdf7b0f9131c): Maximum turns (2) reached

********************************************************************************
Starting a new chat....

********************************************************************************
onboarding_topic_preference_agent (to customer_proxy_agent):

Great! Could you tell me what topics, you are interested in reading about?
Context: 
CUSTOMER_DATA:{"name": "Raji", "location": "Chennai"}


--------------------------------------------------------------------------------


Replying as customer_proxy_agent. Provide feedback to onboarding_topic_preference_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  Thiriller


customer_proxy_agent (to onboarding_topic_preference_agent):

Thiriller

--------------------------------------------------------------------------------
onboarding_topic_preference_agent (to customer_proxy_agent):

TOPIC_PREFERENCE: Thriller
TERMINATE

--------------------------------------------------------------------------------


Replying as customer_proxy_agent. Provide feedback to onboarding_topic_preference_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  



>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> TERMINATING RUN (3975f4fe-b0c8-41de-a8b9-a35155ad0d36): Termination message condition on agent 'customer_proxy_agent' met

>>>>>>>> TERMINATING RUN (b803e40e-2e0e-46e8-8049-3b329ea10502): Maximum turns (2) reached

********************************************************************************
Starting a new chat....

********************************************************************************
customer_proxy_agent (to customer_engagement_agent):

Use the customer information and topic preference
           collected in the previous conversations.
           Generate personalized content based on those details.
           Do not use default or invented customer information.
Context: 
CUSTOMER_DATA:{"name": "Raji", "location": "Chennai"}

TOPIC_PREFERENCE: Thriller


--------------------------------------------------------------------------------
customer_engagement_agent (to customer_proxy_agent):

Hello Raji,

I've got some exciting new

In [10]:
# Extract summaries

summary = []

for chat_result in chat_results:
    print(chat_result.summary)
    summary.append(chat_result.summary)
    print("\n")

CUSTOMER_DATA:{"name": "Raji", "location": "Chennai"}



TOPIC_PREFERENCE: Thriller



Hello Raji,

I've got some exciting news for you! Since your topic preference is Thriller, I'd like to share a short and thrilling story with you.

The Vanishing Hotel

In the heart of Chennai, there was a hotel that seemed to appear out of nowhere. The hotel's name was "Midnight Mirage," and it was shrouded in mystery. Guests who checked in never checked out, but they never spoke of their experiences either.

One stormy night, a young traveler named Rohan stumbled upon the hotel while seeking refuge from the rain. As he entered the lobby, he felt an eerie chill run down his spine. The receptionist, a woman with piercing green eyes, greeted him with a warm smile.

"Welcome to Midnight Mirage," she said. "You'll be staying with us for a while."

Rohan tried to leave the next morning, but the doors were locked. He was trapped. As he explored the hotel, he discovered rooms that seemed to shift and chang

In [11]:
# Convert customer information into a DataFrame

import re

# Get customer information from the first summary
data_str = summary[0]

# Extract the JSON object from the agent's response
match = re.search(r'\{.+\}', data_str, re.DOTALL)

if match:
    data_dict = json.loads(match.group())
else:
    raise ValueError(f"No JSON customer data found in: {data_str}")

# Create DataFrame
df = pd.DataFrame([data_dict])

# Add topic preference
df["TopicPreference"] = summary[1]

# Add customer engagement information
df["Engagement"] = summary[2]

df

,name,location,TopicPreference,Engagement
0,Raji,Chennai,TOPIC_PREFERENCE: Thriller\n,"Hello Raji,\n\nI've got some exciting news for..."


In [12]:
# Display final customer information

print("Customer Onboarding Details")

display(df)

Customer Onboarding Details


,name,location,TopicPreference,Engagement
0,Raji,Chennai,TOPIC_PREFERENCE: Thriller\n,"Hello Raji,\n\nI've got some exciting news for..."


In [13]:
# Display execution cost

for chat_result in chat_results:
    print(chat_result.cost)
    print("\n")

{'usage_including_cached_inference': {'total_cost': 0.0, 'llama3.2': {'cost': 0.0, 'prompt_tokens': 139, 'completion_tokens': 22, 'total_tokens': 161}}, 'usage_excluding_cached_inference': {'total_cost': 0.0, 'llama3.2': {'cost': 0.0, 'prompt_tokens': 139, 'completion_tokens': 22, 'total_tokens': 161}}}


{'usage_including_cached_inference': {'total_cost': 0.0, 'llama3.2': {'cost': 0.0, 'prompt_tokens': 157, 'completion_tokens': 12, 'total_tokens': 169}}, 'usage_excluding_cached_inference': {'total_cost': 0.0, 'llama3.2': {'cost': 0.0, 'prompt_tokens': 157, 'completion_tokens': 12, 'total_tokens': 169}}}


{'usage_including_cached_inference': {'total_cost': 0.0, 'llama3.2': {'cost': 0.0, 'prompt_tokens': 222, 'completion_tokens': 461, 'total_tokens': 683}}, 'usage_excluding_cached_inference': {'total_cost': 0.0, 'llama3.2': {'cost': 0.0, 'prompt_tokens': 222, 'completion_tokens': 461, 'total_tokens': 683}}}


